# YZ50 - 1. Hafta Görevleri: Yapay Sinir Ağları ve Gradient Descent

## Giriş ve Öğrenim Hedefleri
Bu Jupyter Notebook çalışmasında, Yapay Sinir Ağlarının (Artificial Neural Networks) en temel bileşenlerini **hiçbir kütüphane (PyTorch, TensorFlow vb.) kullanmadan, saf Python** ile adım adım inşa ediyoruz.

### Kaynaklar:
1. **3Blue1Brown** - "But what is a neural network?": [YouTube Video](https://www.youtube.com/watch?v=aircAruvnKk)
2. **3Blue1Brown** - "Gradient descent, how neural networks learn": [YouTube Video](https://www.youtube.com/watch?v=IHZwWFHWa-w)
3. **Andrej Karpathy** - "The spelled-out intro to neural networks" (İlk 19 Dakika - Sayısal Türev): [YouTube Video](https://www.youtube.com/watch?v=VMj-3S1tku0)

---

### Görev Listesi:
1. **Tek Nöron Forward Pass**: Python ile kütüphanesiz tek nöron hesabı.
2. **Katman (Layer) Forward Pass**: Birden fazla nörondan oluşan küçük bir katman kurup forward pass'i buna genişletme.
3. **Kayıp Fonksiyonu (Loss Function)**: MSE (Mean Squared Error) hatası hesaplama.
4. **Manuel Parametre Değişimi & Loss Eğrisi**: Parametreyi manuel değiştirerek Loss eğrisini matplotlib ile çizme.
5. **Sayısal Türev & Gradient Descent**: Sayısal türev (numerical derivative) ile parametreleri güncelleyen basit bir Gradient Descent döngüsü kurma.


In [ ]:
# --- Kütüphane İçe Aktarımları (Imports) ---
import sys
import os

# Eğer seçili Python ortamında matplotlib bulunamazsa, sistemdeki Python 3.13 paket yolunu otomatik ekliyoruz:
sys_path_313 = "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages"
if sys_path_313 not in sys.path and os.path.exists(sys_path_313):
    sys.path.append(sys_path_313)

try:
    import matplotlib.pyplot as plt
    print("Matplotlib kütüphanesi başarıyla içe aktarıldı!")
except ModuleNotFoundError:
    print("Matplotlib bulunamadı. Lütfen terminalde veya notebook hücresinde '%pip install matplotlib' çalıştırın.")

import math


## Görev 1: Tek Nöron Forward Pass (Kütüphanesiz)
Bir nöron, kendisine gelen girdileri ($x_1, x_2, \dots, x_n$) ilgili ağırlıklar ($w_1, w_2, \dots, w_n$) ile çarpar, hepsini toplar ve bir sapma ($bias, b$) ekler:

$$z = \sum_{i=1}^{n} (w_i \cdot x_i) + b = (w_1 x_1 + w_2 x_2 + \dots + w_n x_n) + b$$

Aşağıdaki kod bloğunda herhangi bir dış kütüphane kullanmadan, saf Python listeleri ve döngüleri ile hem fonksiyonel hem de Nesne Yönelimli (OOP) olarak nöron yapısını tanımlıyoruz.


In [ ]:
def single_neuron_forward(inputs, weights, bias):
    """
    Tek bir nöronun forward pass (ileri besleme) hesabını yapar.
    """
    return sum(w * x for w, x in zip(weights, inputs)) + bias

# --- NESNE YÖNELİMLİ (OOP) TEK NÖRON SINIFI ---
class Neuron:
    """
    Saf Python ile Nesne Yönelimli (OOP) Tek Nöron Sınıfı.
    """
    def __init__(self, weights, bias):
        self.weights = list(weights)
        self.bias = float(bias)
        
    def forward(self, inputs):
        """Forward pass hesabı: z = sum(w_i * x_i) + b"""
        return sum(w * x for w, x in zip(self.weights, inputs)) + self.bias
        
    def __repr__(self):
        return f"Neuron(weights={self.weights}, bias={self.bias})"

# --- Test Senaryosu ---
x_test = [2.0, 3.0]
w_test = [0.8, -0.5]
b_test = 1.0

# 1. Fonksiyonel Yaklaşım
output_func = single_neuron_forward(x_test, w_test, b_test)

# 2. OOP Class Yaklaşımı
neuron_obj = Neuron(w_test, b_test)
output_oop = neuron_obj.forward(x_test)

print("--- Görev 1: Tek Nöron Forward Pass ---")
print(f"Girdiler (x)                  : {x_test}")
print(f"Ağırlıklar (w)               : {w_test}")
print(f"Bias (b)                     : {b_test}")
print(f"Fonksiyonel Çıktı (single_fn): {output_func:.4f}")
print(f"OOP Class Çıktısı (Neuron)   : {output_oop:.4f}")


## Görev 2: Birden Fazla Nöronlu Katman (Layer Forward Pass)
Bir katman (Layer), aynı girdileri paralel olarak işleyen birden fazla nörondan oluşur.

Örneğin 2 girdisi olan ve 3 nörondan oluşan bir katmanda:
- **1. Nöron**: Kendi ağırlıkları $W^{(1)}$ ve bias'ı $b_1$ ile çıktı üretir ($z_1$).
- **2. Nöron**: Kendi ağırlıkları $W^{(2)}$ ve bias'ı $b_2$ ile çıktı üretir ($z_2$).
- **3. Nöron**: Kendi ağırlıkları $W^{(3)}$ ve bias'ı $b_3$ ile çıktı üretir ($z_3$).

Katmanın çıktısı, tüm nöronların çıktılarından oluşan bir vektördür (liste): $[z_1, z_2, z_3]$.


In [ ]:
def layer_forward(inputs, weights_list, biases):
    """
    Birden fazla nörondan oluşan bir katmanın forward pass hesabını yapar.
    """
    outputs = []
    for weights, bias in zip(weights_list, biases):
        neuron_out = single_neuron_forward(inputs, weights, bias)
        outputs.append(neuron_out)
    return outputs

# --- NESNE YÖNELİMLİ (OOP) KATMAN (LAYER) SINIFI ---
class Layer:
    """
    Saf Python ile Nesne Yönelimli (OOP) Katman Sınıfı.
    Birden fazla Neuron nesnesini bünyesinde barındırır.
    """
    def __init__(self, neurons=None):
        self.neurons = list(neurons) if neurons else []
        
    def add_neuron(self, neuron):
        self.neurons.append(neuron)
        
    def forward(self, inputs):
        """Katmandaki tüm nöronlar için ileri besleme çalıştırır."""
        return [neuron.forward(inputs) for neuron in self.neurons]
        
    def __repr__(self):
        return f"Layer(num_neurons={len(self.neurons)})"

# --- Test Senaryosu ---
inputs = [2.0, 3.0]
weights_list = [
    [0.8, -0.5],   # 1. Nöron
    [0.2, 0.9],    # 2. Nöron
    [-0.4, 0.6]    # 3. Nöron
]
biases = [1.0, -0.5, 0.2]

# 1. Fonksiyonel Katman Çıktısı
layer_outputs_func = layer_forward(inputs, weights_list, biases)

# 2. OOP Layer Class Çıktısı
neurons = [Neuron(w, b) for w, b in zip(weights_list, biases)]
layer_obj = Layer(neurons)
layer_outputs_oop = layer_obj.forward(inputs)

print("--- Görev 2: Katman Forward Pass ---")
print(f"Ortak Girdiler                 : {inputs}")
print(f"Fonksiyonel Katman Çıktısı     : {layer_outputs_func}")
print(f"OOP Layer Class Çıktısı        : {layer_outputs_oop}")
print(f"Oluşturulan Katman Nesnesi    : {layer_obj}")


## Görev 3: Basit Bir Loss Fonksiyonu (MSE - Mean Squared Error)
Modelin ürettiği tahmin ($y_{pred}$) ile ulaşmak istediğimiz gerçek hedef ($y_{true}$) arasındaki hatayı/farkı ölçmek için **Loss (Kayıp) Fonksiyonu** kullanılır.

### 1. Tek Nöron İçin Loss (Kare Hata):
$$L_{single} = (y_{pred} - y_{true})^2$$

### 2. Birden Fazla Nöron (Katman) İçin Loss (MSE):
Katmandaki $N$ adet nöronun ürettiği tahmin vektörü $[y_{pred}^{(1)}, y_{pred}^{(2)}, \dots, y_{pred}^{(N)}]$ ve hedef vektörü $[y_{true}^{(1)}, y_{true}^{(2)}, \dots, y_{true}^{(N)}]$ için Ortalama Kare Hata (MSE):

$$L_{layer} = \frac{1}{N} \sum_{i=1}^{N} (y_{pred}^{(i)} - y_{true}^{(i)})^2$$

Aşağıdaki kod bloğunda hem **tek nöron** hem de **birden fazla nöron (katman)** için kayıp fonksiyonlarını ayrı ayrı ve genel olarak yazıyoruz.


In [ ]:
# 1. Tek Nöron İçin Loss Fonksiyonu
def single_neuron_loss(y_pred, y_true):
    """Tek bir nöron çıktısı için Kare Hata (Squared Error) hesaplar."""
    return (y_pred - y_true) ** 2

# 2. Birden Fazla Nöron (Katman) İçin Loss Fonksiyonu
def multi_neuron_layer_loss(y_preds, y_trues):
    """
    Katmandaki birden fazla nöron çıktısı için Ortalama Kare Hata (MSE) hesaplar.
    
    Parametreler:
    - y_preds: Katmandaki tüm nöronların tahmin listesi [z1, z2, ...]
    - y_trues: Her nöron için beklenen hedef liste [t1, t2, ...]
    """
    total_squared_error = sum((p - t) ** 2 for p, t in zip(y_preds, y_trues))
    return total_squared_error / len(y_trues)

# 3. Genel Esnek Loss Fonksiyonu (Hem Tek Hem Çoklu Nöron Destekli)
def mse_loss(y_pred, y_true):
    """Hem skaler (tek nöron) hem liste (çoklu nöron/katman) destekleyen esnek MSE fonksiyonu."""
    if isinstance(y_pred, (int, float)) and isinstance(y_true, (int, float)):
        return single_neuron_loss(y_pred, y_true)
    return multi_neuron_layer_loss(y_pred, y_true)

# --- Test Senaryoları ---

# A) TEK NÖRON İÇİN LOSS HESABI
pred_single = single_neuron_forward([2.0, 3.0], [0.8, -0.5], 1.0) # 1.1
target_single = 5.0

loss_single_explicit = single_neuron_loss(pred_single, target_single)

print("--- Görev 3: Loss (Kayıp) Hesaplama ---")
print("[1] TEK NÖRON İÇİN LOSS HESABI:")
print(f"    Tek Nöron Tahmini (y_pred) : {pred_single:.4f}")
print(f"    Gerçek Hedef (y_true)      : {target_single:.4f}")
print(f"    Kayıp (single_neuron_loss)  : {loss_single_explicit:.4f}")
print()

# B) BİRDEN FAZLA NÖRON (KATMAN) İÇİN LOSS HESABI
inputs = [2.0, 3.0]
weights_list = [
    [0.8, -0.5],   # 1. Nöron
    [0.2, 0.9],    # 2. Nöron
    [-0.4, 0.6]    # 3. Nöron
]
biases = [1.0, -0.5, 0.2]

# 3 Nöronlu Katmanın Tahminleri
layer_preds = layer_forward(inputs, weights_list, biases) # [1.1, 2.6, 1.2]
# 3 Nöron için 3 Ayrı Gerçek Hedef
target_layer = [5.0, 3.0, 2.0]

loss_layer_explicit = multi_neuron_layer_loss(layer_preds, target_layer)

print("[2] BİRDEN FAZLA NÖRON (KATMAN) İÇİN LOSS HESABI:")
print(f"    Katman Tahminleri (y_preds)      : {[round(p, 4) for p in layer_preds]}")
print(f"    Gerçek Hedefler (y_trues)        : {target_layer}")
print(f"    Kayıp (multi_neuron_layer_loss) : {loss_layer_explicit:.4f}")


## Görev 4: Manuel Parametre Değişimi ve Loss Eğrisinin Çizilmesi
3Blue1Brown'ın 2. videosundaki sezgiye göre:
- Modelin parametrelerinden birini ($w_1$) manuel olarak adımlarla değiştiriyoruz.
- Diğer parametreleri ($w_2, b$) ve girdileri ($x_1, x_2$) sabit tutuyoruz.
- Her $w_1$ değeri için forward pass çalıştırıp Loss hesaplıyoruz.
- Elde ettiğimiz $(w_1, Loss)$ çiftlerini grafik üzerinde çizerek **Loss Eğrisini (Loss Landscape)** inceliyoruz.


In [ ]:
import matplotlib.pyplot as plt

# Sabit Değerler
x_input = [2.0, 3.0]
w2_fixed = -0.5
b_fixed = 1.0
target = 10.0  # Ulaşmak istediğimiz hedef değer

# w1 parametresini -5.0 ile +8.0 arasında küçük adımlarla değiştiriyoruz
w1_range = [w1_int / 10.0 for w1_int in range(-50, 81)] # [-5.0, -4.9, ..., 8.0]
loss_list = []

for w1 in w1_range:
    # Forward Pass
    current_pred = single_neuron_forward(x_input, [w1, w2_fixed], b_fixed)
    # Loss Hesabı
    current_loss = mse_loss(current_pred, target)
    loss_list.append(current_loss)

# Loss Eğrisini Çizdirelim
plt.figure(figsize=(9, 5))
plt.plot(w1_range, loss_list, color='#e74c3c', linewidth=2.5, label='MSE Loss')
plt.title("Görev 4: w1 Parametresinin Değişimine Göre Loss Eğrisi", fontsize=13, fontweight='bold')
plt.xlabel("w1 Ağırlık Değeri", fontsize=11)
plt.ylabel("Loss (Hata Değeri)", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

# Minimum noktasını vurgulayalım
min_loss = min(loss_list)
best_w1 = w1_range[loss_list.index(min_loss)]
plt.axvline(x=best_w1, color='#2ecc71', linestyle=':', linewidth=2, label=f'En Düşük Loss Noktası (w1 ≈ {best_w1:.2f})')

plt.legend(fontsize=10)
plt.show()

print("--- Görev 4 Sonucu ---")
print(f"Gözlemlenen en düşük loss değeri: {min_loss:.4f} (w1 = {best_w1:.2f} için)")


## Görev 5: Sayısal Türev (Numerical Derivative) ile Gradient Descent Döngüsü
Andrej Karpathy'nin ilk 19 dakikasında tarif ettiği gibi, bir fonksiyonun türevi sayısal olarak şöyle hesaplanabilir:

$$f'(x) \approx \frac{f(x + h) - f(x)}{h}$$

Burada $h$ çok küçük bir adımdır (örneğin $h = 10^{-5}$).

Yapay sinir ağımızda kayıp fonksiyonumuz $L(w_1, w_2, b)$ parametrelerimize bağlıdır. Her bir parametrenin kayıp üzerindeki **kısmi türevini (gradyanını)** sayısal olarak hesaplarız:

$$\frac{\partial L}{\partial w_1} \approx \frac{L(w_1 + h, w_2, b) - L(w_1, w_2, b)}{h}$$

### Gradient Descent Güncelleme Kuralı:
Gradyan bize hatanın **en hızlı arttığı** yönü gösterir. Hatayı **düşürmek** için gradyanın **tersi yönünde** adım atarız:

$$w_1 \leftarrow w_1 - \eta \cdot \frac{\partial L}{\partial w_1}$$
$$w_2 \leftarrow w_2 - \eta \cdot \frac{\partial L}{\partial w_2}$$
$$b \leftarrow b - \eta \cdot \frac{\partial L}{\partial b}$$

Burada $\eta$ (eta) öğrenme oranıdır (Learning Rate).


In [ ]:
def compute_numerical_gradients(inputs, weights, bias, y_true, h=1e-5):
    """
    Sayısal türev (Numerical Derivative) kullanarak her parametrenin
    Loss üzerindeki kısmi türevini (gradyanını) hesaplar.
    """
    # 1. Mevcut parametrelerle loss hesabı
    base_pred = single_neuron_forward(inputs, weights, bias)
    base_loss = mse_loss(base_pred, y_true)
    
    # 2. Her bir ağırlık için gradyan hesabı: dL/dwi
    grad_weights = []
    for i in range(len(weights)):
        weights_perturbed = list(weights)
        weights_perturbed[i] += h  # wi parametresine küçük h ekliyoruz
        
        pred_perturbed = single_neuron_forward(inputs, weights_perturbed, bias)
        loss_perturbed = mse_loss(pred_perturbed, y_true)
        
        # Sayısal türev formülü: (L(w + h) - L(w)) / h
        dw_i = (loss_perturbed - base_loss) / h
        grad_weights.append(dw_i)
        
    # 3. Bias için gradyan hesabı: dL/db
    bias_perturbed = bias + h
    pred_b_perturbed = single_neuron_forward(inputs, weights, bias_perturbed)
    loss_b_perturbed = mse_loss(pred_b_perturbed, y_true)
    db = (loss_b_perturbed - base_loss) / h
    
    return grad_weights, db, base_loss

# --- Gradient Descent Döngüsü ---

# Problem Tanımlaması
inputs = [2.0, 3.0]
y_true = 10.0  # Hedef Çıktı

# Başlangıç Parametreleri (Rastgele / Sıfır başlatma)
weights = [0.0, 0.0]
bias = 0.0

learning_rate = 0.02  # Adım boyutu (Learning Rate)
epochs = 40           # İterasyon sayısı

history_loss = []
history_w1 = []

print("--- Görev 5: Gradient Descent Eğitimi Başlıyor ---")
print(f"Başlangıç Parametreleri -> w: {weights}, b: {bias}")
print(f"Başlangıç Tahmini       -> {single_neuron_forward(inputs, weights, bias):.4f} (Hedef: {y_true})\n")

for epoch in range(epochs):
    # Sayısal gradyanları hesapla
    grad_w, grad_b, current_loss = compute_numerical_gradients(inputs, weights, bias, y_true)
    
    # Takip için kayıt
    history_loss.append(current_loss)
    history_w1.append(weights[0])
    
    # Parametreleri Gradyan Yönünün TERSİNE Güncelle
    for i in range(len(weights)):
        weights[i] -= learning_rate * grad_w[i]
    bias -= learning_rate * grad_b
    
    # Durum çıktısı
    if epoch % 5 == 0 or epoch == epochs - 1:
        pred = single_neuron_forward(inputs, weights, bias)
        print(f"Epoch {epoch:2d} | Loss: {current_loss:8.4f} | Tahmin: {pred:6.4f} | w1: {weights[0]:.4f}, w2: {weights[1]:.4f}, b: {bias:.4f}")

final_pred = single_neuron_forward(inputs, weights, bias)
print("\n--- Eğitim Tamamlandı ---")
print(f"Son Parametreler -> w1: {weights[0]:.4f}, w2: {weights[1]:.4f}, b: {bias:.4f}")
print(f"Son Tahmin      -> {final_pred:.4f} (Hedef: {y_true})")
print(f"Son Loss        -> {mse_loss(final_pred, y_true):.6f}")

# Gradient Descent Görselleştirmesi
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Loss'un Adımlarla Düşüşü
ax1.plot(range(epochs), history_loss, color='#2980b9', marker='o', markersize=4, linewidth=2)
ax1.set_title("Gradient Descent: Loss'un Düşüş Grafiği", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch (Adım)", fontsize=10)
ax1.set_ylabel("MSE Loss", fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.6)

# 2. w1 Parametresinin Güncellenmesi
ax2.plot(range(epochs), history_w1, color='#8e44ad', marker='s', markersize=4, linewidth=2)
ax2.set_title("Gradient Descent: w1 Parametresinin İlerlemesi", fontsize=12, fontweight='bold')
ax2.set_xlabel("Epoch (Adım)", fontsize=10)
ax2.set_ylabel("w1 Ağırlık Değeri", fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## Özet ve Genel Değerlendirme

Bu ödev çalışmasında yapay sinir ağlarının ve makine öğrenmesinin temellerini saf Python ile sıfırdan inşa ettik:

1. **Forward Pass (İleri Besleme)**: Girdilerin parametreler (w, b) ile çarpılıp toplanarak tahmin oluşturması.
2. **Katman Mimarisi**: Nöronların bir araya gelerek katman oluşturması ve çıktılar üretmesi.
3. **Loss (Kayıp)**: Model tahmininin kalitesini ölçen matematiksel hedef fonksiyonu (MSE).
4. **Manuel Parametre Analizi**: Parametrelerin değişimine karşılık Loss fonksiyonunun konveks yapısının gözlemlenmesi.
5. **Modelin Öğrenmesi (Gradient Descent)**: Hatayı minimize etmek için sayısal türev (numerical derivative) kullanılarak gradyanların hesaplanması ve parametrelerin hatayı düşürecek yönde otomatik güncellenmesi.

Böylece "Modelin Öğrenmesi" kavramının arkasındaki temel matematiksel ve kodlama mantığını kütüphanesiz olarak kavramış olduk!
